<a href="https://colab.research.google.com/github/Lungile-Hlengane/VWAP-execution-simulator/blob/main/shape_cache_backfill_and_app_(4).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BTC/USDT VWAP Shape Cache — Backfill, Incremental Update, and App Integration

This notebook replaces the "recompute every prior day's hourly volume shape from
full tick data, on every single request" pattern with a small cached JSON file.

**What's in here:**
1. Config + shared helpers (S3 tick loading, shape computation, cache read/write)
2. One-time **backfill** — computes the 24-value hourly shape for every existing
   day and writes them all into `parquets/shape_cache/hourly_shapes.json`
3. The **app-side** function (`run_single_day_cached`) showing how `app.py` should
   use the cache instead of recomputing shapes from scratch
4. An **incremental update** step to keep the cache fresh as new days land in S3
   (run manually, or on a schedule)

Run section 1 first always. Run section 2 once (or whenever you want to force a
full rebuild). Sections 3 and 4 are what actually goes into your Streamlit app /
scheduled job, respectively — they're runnable here too so you can sanity-check
them before copying them over.


## 1. Setup

In [1]:
!pip install pandas numpy s3fs pyarrow -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.5/89.5 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.9/203.9 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.0/15.0 MB 43.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.6.0 which is incompatible.
datasets 4.0.0 requires fsspec[http]<=2025.3.0,>=2023.1.0, but you have fsspec 2026.6.0 which is incompatible.


In [2]:
import glob
import json
import os
import re
from datetime import datetime, timedelta

import numpy as np
import pandas as pd
import s3fs


### Config

Point this at your actual bucket/prefixes. Everything downstream assumes these
S3 paths — the tick data as it's uploaded today, and the (new) small shape
cache file this notebook creates.


In [3]:
S3_BUCKET = os.environ.get("S3_BUCKET", "madot-algo-data")

AGGTRADES_PREFIX = "parquets/aggTrades"
SHAPE_CACHE_KEY  = "parquets/shape_cache/hourly_shapes.json"

AGGTRADES_S3_PATH   = f"s3://{S3_BUCKET}/{AGGTRADES_PREFIX}"
SHAPE_CACHE_S3_PATH = f"s3://{S3_BUCKET}/{SHAPE_CACHE_KEY}"

# Keep these in sync with whatever app.py already uses for the execution window.
START_TIME_STR = "00:00:00"
END_TIME_STR   = "23:59:59"

DATE_RE = re.compile(r"(\d{4}-\d{2}-\d{2})")



### AWS credentials

`NoCredentialsError` means boto3/s3fs couldn't find AWS credentials in this
environment (this is expected the first time you run this in a fresh Colab
runtime - Colab has no AWS config by default).

Pick **one** of the following, then run the cell below it:

- **Recommended for Colab**: store your key/secret as Colab secrets (key icon
  in the left sidebar) named `AWS_ACCESS_KEY_ID` and `AWS_SECRET_ACCESS_KEY`,
  then run the cell as-is - it will pick them up automatically.
- **One-off / other environments**: leave the secrets unset and the cell will
  fall back to prompting you for the key/secret directly (hidden input, not
  stored to disk).
- **Already have a shared AWS profile / IAM role configured** (e.g. running
  this on an EC2 instance or you've set up `~/.aws/credentials`): skip this
  cell entirely, `s3fs.S3FileSystem()` will pick that up on its own.


In [4]:
import os
from getpass import getpass

try:
    from google.colab import userdata
    _access_key = userdata.get("AWS_ACCESS_KEY_ID")
    _secret_key = userdata.get("AWS_SECRET_ACCESS_KEY")
except Exception:
    _access_key = os.environ.get("AWS_ACCESS_KEY_ID")
    _secret_key = os.environ.get("AWS_SECRET_ACCESS_KEY")

if not _access_key:
    _access_key = getpass("Enter your AWS Access Key ID: ")
if not _secret_key:
    _secret_key = getpass("Enter your AWS Secret Access Key: ")

os.environ["AWS_ACCESS_KEY_ID"] = _access_key
os.environ["AWS_SECRET_ACCESS_KEY"] = _secret_key
os.environ.setdefault("AWS_DEFAULT_REGION", "eu-north-1")  # madot-algo-data lives in Stockholm (eu-north-1)

del _access_key, _secret_key
print("AWS credentials set for this session.")


Enter your AWS Access Key ID: ··········
Enter your AWS Secret Access Key: ··········
AWS credentials set for this session.


In [5]:
# Create the S3 filesystem handle AFTER credentials are set above.
fs = s3fs.S3FileSystem()


### Shared helpers

`load_day_trades` and `get_hourly_volume_shape` are unchanged in *behavior*
from the original notebook — just pointed at S3 instead of Google Drive.
Everything else here (`list_available_days`, `load_shape_cache`,
`save_shape_cache`, `compute_shape_for_day`) is new — this is the caching
layer itself.


In [6]:
def load_day_trades(date_str, base_path=AGGTRADES_S3_PATH):
    """
    Loads ONLY 1 day of BTCUSDT aggTrades tick data from S3.
    date_str format: '2026-06-05'

    This should only ever be called for the *target* simulation date once the
    shape cache is in place - every prior day's shape should come from the
    cache instead (see run_single_day_cached in section 3).
    """
    pattern = f"{base_path}/*{date_str}.parquet"
    files = sorted(fs.glob(pattern))
    if not files:
        print(f"No files found for {date_str}")
        return pd.DataFrame()

    dfs = []
    for f in files:
        df = pd.read_parquet(
            f"s3://{f}",
            columns=["price", "quantity", "transact_time", "is_buyer_maker"],
        )
        for col in df.select_dtypes(include=["float64"]).columns:
            df[col] = pd.to_numeric(df[col], downcast="float")
        for col in df.select_dtypes(include=["int64"]).columns:
            df[col] = pd.to_numeric(df[col], downcast="unsigned")

        df["transact_time"] = pd.to_datetime(df["transact_time"], unit="ms")
        dfs.append(df)

    df_day = pd.concat(dfs, ignore_index=True)
    df_day = df_day.drop_duplicates(subset=["transact_time", "price", "quantity"])
    print(
        f"Loaded {date_str}: {len(df_day):,} rows | "
        f"RAM: ~{df_day.memory_usage(deep=True).sum() / 1024**2:.1f} MB"
    )
    return df_day


def get_hourly_volume_shape(df_day, start_time_str, end_time_str):
    """Returns 24 hourly % shares of volume (coarse - stable for GA)."""
    day_date = df_day["transact_time"].dt.date.iloc[0]
    window_start = pd.Timestamp(f"{day_date} {start_time_str}")
    window_end   = pd.Timestamp(f"{day_date} {end_time_str}")
    window = df_day[(df_day["transact_time"] >= window_start) &
                     (df_day["transact_time"] <= window_end)]

    hourly_edges = pd.date_range(window_start, window_end, periods=25)
    total_qty = window["quantity"].sum()
    if total_qty == 0:
        return np.full(24, np.nan)

    shape = []
    for i in range(24):
        slice_qty = window[(window["transact_time"] >= hourly_edges[i]) &
                            (window["transact_time"] < hourly_edges[i + 1])]["quantity"].sum()
        shape.append(slice_qty / total_qty)
    return np.array(shape)


In [7]:
def list_available_days(base_path=AGGTRADES_S3_PATH):
    """Sorted list of 'YYYY-MM-DD' strings for every day with tick data in S3."""
    files = fs.glob(f"{base_path}/*.parquet")
    days = set()
    for f in files:
        m = DATE_RE.search(os.path.basename(f))
        if m:
            days.add(m.group(1))
    return sorted(days)


def load_shape_cache(path=SHAPE_CACHE_S3_PATH):
    """Loads the shape cache JSON. Returns {} if it doesn't exist yet."""
    key = path.replace("s3://", "")
    if not fs.exists(key):
        return {}
    with fs.open(key, "r") as f:
        return json.load(f)


def save_shape_cache(cache_dict, path=SHAPE_CACHE_S3_PATH):
    """Writes the full {date_str: [24 floats]} cache back to S3 as one file."""
    key = path.replace("s3://", "")
    with fs.open(key, "w") as f:
        json.dump(cache_dict, f)
    print(f"Wrote {len(cache_dict)} day-shapes to s3://{key}")


def compute_shape_for_day(date_str, start_time_str=START_TIME_STR, end_time_str=END_TIME_STR):
    """Slow path: loads one day's tick data and computes its shape.
    Returns None if the day has no usable data."""
    df_day = load_day_trades(date_str)
    if df_day.empty:
        return None
    shape = get_hourly_volume_shape(df_day, start_time_str, end_time_str)
    del df_day
    if np.isnan(shape).all():
        return None
    return shape


## 2. One-time backfill

Computes the hourly shape for every day currently in S3 and writes them all
into a single small `hourly_shapes.json`. Safe to re-run — by default it
skips days already in the cache; pass `overwrite=True` to force a full rebuild.


In [8]:
def backfill_shape_cache(overwrite=False):
    print("Discovering available days in S3...")
    all_days = list_available_days()
    if not all_days:
        print("No tick data found - check AGGTRADES_S3_PATH.")
        return {}
    print(f"Found {len(all_days)} days of tick data: {all_days[0]} .. {all_days[-1]}")

    cache = {} if overwrite else load_shape_cache()
    print(f"Existing cache has {len(cache)} entries.")

    computed, skipped, failed = 0, 0, 0
    for date_str in all_days:
        if not overwrite and date_str in cache:
            skipped += 1
            continue

        print(f"Computing shape for {date_str}...")
        shape = compute_shape_for_day(date_str)
        if shape is None:
            print(f"  -> no usable data for {date_str}, skipping")
            failed += 1
            continue

        cache[date_str] = shape.tolist()
        computed += 1

    print(
        f"\nDone. Computed {computed} new day(s), skipped {skipped} already-cached, "
        f"{failed} had no usable data."
    )
    save_shape_cache(cache)
    return cache


# Run the backfill. Comment this out after the first successful run if you'd
# rather trigger it explicitly.
shape_cache = backfill_shape_cache(overwrite=False)


Discovering available days in S3...
Found 91 days of tick data: 2026-04-01 .. 2026-06-30
Existing cache has 0 entries.
Computing shape for 2026-04-01...
Loaded 2026-04-01: 1,591,260 rows | RAM: ~44.0 MB
Computing shape for 2026-04-02...
Loaded 2026-04-02: 1,834,455 rows | RAM: ~50.7 MB
Computing shape for 2026-04-03...
Loaded 2026-04-03: 797,427 rows | RAM: ~22.1 MB
Computing shape for 2026-04-04...
Loaded 2026-04-04: 378,310 rows | RAM: ~10.5 MB
Computing shape for 2026-04-05...
Loaded 2026-04-05: 809,832 rows | RAM: ~22.4 MB
Computing shape for 2026-04-06...
Loaded 2026-04-06: 1,617,381 rows | RAM: ~44.7 MB
Computing shape for 2026-04-07...
Loaded 2026-04-07: 2,115,330 rows | RAM: ~58.5 MB
Computing shape for 2026-04-08...
Loaded 2026-04-08: 1,632,550 rows | RAM: ~45.2 MB
Computing shape for 2026-04-09...
Loaded 2026-04-09: 1,700,074 rows | RAM: ~47.0 MB
Computing shape for 2026-04-10...
Loaded 2026-04-10: 1,417,174 rows | RAM: ~39.2 MB
Computing shape for 2026-04-11...
Loaded 2026-0

## 3. App-side usage: `run_single_day_cached`

This is what `app.py`'s simulation entry point should look like. Compare to
the old version: instead of walking every prior day and calling
`load_day_trades()` + `get_hourly_volume_shape()` on each one, it:

1. Loads the single small cache file (in Streamlit this line would be wrapped
   in `@st.cache_data(ttl=3600)` so it only hits S3 once per session/hour,
   not once per request)
2. Looks up each prior day's shape from the cache
3. Falls back to computing + caching **live** only for the handful of days
   not yet backfilled (e.g. the most recent day or two)
4. Still uses `load_day_trades()` for full tick data — but *only* for the
   target date, since that's the only day the fill simulation actually needs
   tick-level detail for


In [9]:
def run_single_day_cached(target_date_str, capture_pct=0.60, history_start="2026-04-01"):
    target_date = datetime.strptime(target_date_str, "%Y-%m-%d")

    # --- In app.py, replace this line with a Streamlit-cached loader, e.g.:
    #
    #   @st.cache_data(ttl=3600)
    #   def _load_shape_cache_cached():
    #       return load_shape_cache()
    #
    #   cache = _load_shape_cache_cached()
    #
    # so this file is only actually fetched from S3 once per hour/session,
    # not once per simulation request.
    cache = load_shape_cache()

    shape_history = []
    newly_computed = {}  # any live-computed shapes get persisted back at the end

    d = datetime.strptime(history_start, "%Y-%m-%d")
    while d < target_date:
        date_str = d.strftime("%Y-%m-%d")

        if date_str in cache:
            shape = np.array(cache[date_str])
        else:
            # Not backfilled yet (e.g. a very recent day) - compute it live,
            # but only for this one day, not the whole history.
            shape = compute_shape_for_day(date_str)
            if shape is None:
                d += timedelta(days=1)
                continue
            newly_computed[date_str] = shape.tolist()

        shape_history.append(shape)
        d += timedelta(days=1)

    print(f"Built history from {len(shape_history)} prior days "
          f"({len(newly_computed)} computed live, rest from cache)")

    # Persist any newly-computed shapes so future requests never recompute this day.
    if newly_computed:
        cache.update(newly_computed)
        save_shape_cache(cache)

    if len(shape_history) < MIN_LOOKBACK_DAYS:
        print(f"Not enough history yet ({len(shape_history)} days) - need at least {MIN_LOOKBACK_DAYS}")
        return pd.DataFrame()

    # Only the target date needs full tick-level data, for the actual fill simulation.
    df_target = load_day_trades(target_date_str)
    if df_target.empty:
        print(f"No data found for target date {target_date_str}")
        return pd.DataFrame()

    result = your_vwap_strategy(
        df_target, shape_history, TOTAL_ORDER_USD, START_TIME_STR, END_TIME_STR,
        N_TERMINALS, IMPACT_COEFFICIENT, IMPACT_EXPONENT,
        MIN_LOOKBACK_DAYS, MAX_LOOKBACK_DAYS, capture_pct=capture_pct)

    return result

# Example call (uncomment once TOTAL_ORDER_USD / N_TERMINALS / etc. and
# your_vwap_strategy are defined elsewhere in app.py):
# results = run_single_day_cached(TARGET_DATE, capture_pct=0.60)


## 4. Incremental update (keep the cache fresh)

Run this whenever new days of tick data land in S3 — manually, or wire it
into a daily cron / scheduled Lambda / Airflow task shortly after your
aggTrades upload job finishes. It's a no-op if the cache is already
caught up, so it's cheap to run often.

As a standalone script (`update_shape_cache.py`) this would just be:

```bash
5 0 * * * cd /path/to/app && S3_BUCKET=your-bucket python update_shape_cache.py >> update.log 2>&1
```


In [10]:
def update_shape_cache():
    cache = load_shape_cache()
    all_days = list_available_days()

    missing = [d for d in all_days if d not in cache]
    if not missing:
        print("Shape cache already up to date - nothing to do.")
        return cache

    print(f"Found {len(missing)} day(s) missing from cache: {missing}")

    added = 0
    for date_str in missing:
        shape = compute_shape_for_day(date_str)
        if shape is None:
            print(f"  {date_str}: no usable data, will retry next run")
            continue
        cache[date_str] = shape.tolist()
        added += 1
        print(f"  {date_str}: cached")

    if added:
        save_shape_cache(cache)
        print(f"Added {added} new day(s) to the cache.")
    else:
        print("No days had usable data yet; cache not modified.")

    return cache

# Run manually whenever you want to top up the cache:
# shape_cache = update_shape_cache()


## 5. Push this notebook to GitHub

Uploads this notebook to .
You will be prompted for a GitHub Personal Access Token (with repo write access) — it is not stored anywhere.


In [11]:
import requests, base64, json, os
from getpass import getpass

# -- Config --------------------------------------------------------------------
REPO   = 'hlenganeindustries-alt/VWAP-execution-simulator'
BRANCH = 'main'
PATH   = 'shape_cache_backfill_and_app.ipynb'   # path inside the repo

# -- Token input (hidden) --------------------------------------------------------
TOKEN = getpass('Enter your GitHub Personal Access Token: ')

# -- Read this notebook file ------------------------------------------------------
notebook_path = '/content/shape_cache_backfill_and_app.ipynb'
if not os.path.exists(notebook_path):
    # Colab sometimes saves under a different name (e.g. with a '(1)' suffix) - find it
    candidates = [f for f in os.listdir('/content') if f.endswith('.ipynb')]
    if candidates:
        notebook_path = f'/content/{candidates[0]}'
        print(f'Using notebook: {notebook_path}')
    else:
        raise FileNotFoundError('No .ipynb found in /content - save the notebook first (Ctrl+S)')

with open(notebook_path, 'rb') as f:
    content_b64 = base64.b64encode(f.read()).decode()

# -- GitHub API --------------------------------------------------------------------
headers = {
    'Authorization': f'token {TOKEN}',
    'Accept'       : 'application/vnd.github.v3+json',
}
api_url = f'https://api.github.com/repos/{REPO}/contents/{PATH}'

# Check if file already exists (need its SHA to update)
existing = requests.get(api_url, headers=headers, params={'ref': BRANCH})
sha = existing.json().get('sha') if existing.status_code == 200 else None

# Commit message
commit_msg = 'Add shape cache: backfill, app usage, incremental update'

payload = {
    'message': commit_msg,
    'content': content_b64,
    'branch' : BRANCH,
}
if sha:
    payload['sha'] = sha   # required for updates

response = requests.put(api_url, headers=headers, data=json.dumps(payload))

# -- Result --------------------------------------------------------------------
if response.status_code in (200, 201):
    action = 'Updated' if sha else 'Created'
    html_url = response.json()['content']['html_url']
    print(f'✓ {action} successfully!')
    print(f'  {html_url}')
else:
    print(f'✗ Failed ({response.status_code})')
    print(response.json().get('message', response.text))


Enter your GitHub Personal Access Token: ··········


FileNotFoundError: No .ipynb found in /content - save the notebook first (Ctrl+S)